In [1]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import os

In [2]:
# Diccionario mapeando el archivo original con su ruta destino (Gold Zone)
MAP_FILE = {
    "../DATA/Silver/COVID19MEXICO2020/COVID19MEXICO2020.csv": "../DATA/Gold/COVID19MEXICO2020/COVID19MEXICO2020.parquet",
    "../DATA/Silver/COVID19MEXICO2021/COVID19MEXICO2021.csv": "../DATA/Gold/COVID19MEXICO2021/COVID19MEXICO2021.parquet",
    "../DATA/Silver/COVID19MEXICO2022/COVID19MEXICO2022.csv": "../DATA/Gold/COVID19MEXICO2022/COVID19MEXICO2022.parquet",
    "../DATA/Silver/COVID19MEXICO2023/COVID19MEXICO2023.csv": "../DATA/Gold/COVID19MEXICO2023/COVID19MEXICO2023.parquet",
    "../DATA/Silver/COVID19MEXICO2024/COVID19MEXICO2024.csv": "../DATA/Gold/COVID19MEXICO2024/COVID19MEXICO2024.parquet"
}

# La misma definición de dimensiones que usamos en la fase anterior
DIMENSIONS = {
    'DIM_Geografico_informacion_paciente': ['ENTIDAD_UM', 'ENTIDAD_NAC'],
    'DIM_Geografico_residencia': ['ENTIDAD_RES', 'MUNICIPIO_RES'],
    'DIM_Geografico_Nacionalidad': ['NACIONALIDAD', 'PAIS_NACIONALIDAD', 'PAIS_ORIGEN'],
    'DIM_Descripcion_del_paciente': ['SEXO', 'EDAD', 'TIPO_PACIENTE'],
    'DIM_Indigena': ['HABLA_LENGUA_INDIG', 'INDIGENA', 'MIGRANTE'],
    'DIM_Comorbilidades_Respiratorias': ['INTUBADO', 'NEUMONIA', 'EPOC', 'ASMA', 'TABAQUISMO'],
    'DIM_Comorbilidades_de_presion': ['DIABETES', 'INMUSUPR', 'HIPERTENSION', 'CARDIOVASCULAR', 'OBESIDAD'],
    'DIM_Otras_caracteristicas_medicas': ['EMBARAZO', 'RENAL_CRONICA', 'OTRA_COM'],
    'DIM_Ubicacion_de_laboratorio': ['ORIGEN', 'SECTOR', 'OTRO_CASO', 'UCI', 'RESULTADO_PCR', 'RESULTADO_PCR_COINFECCION'],
    'DIM_Antigeno': ['TOMA_MUESTRA_ANTIGENO', 'RESULTADO_ANTIGENO'],
    'DIM_Datos_de_laboratorio': ['TOMA_MUESTRA_LAB', 'RESULTADO_LAB', 'CLASIFICACION_FINAL_COVID', 'CLASIFICACION_FINAL_FLU']
}

DIMENSIONS_PATH = "../TRANSFORM/dimensiones"

In [3]:
def load_dimensions_in_memory():
    """Carga las tablas de dimensiones físicas, conservando solo el ID y las columns clave para el Join."""
    dict_dims = {}
    print("Cargando dimensiones en memoria RAM...")
    
    for name_dim, columns in DIMENSIONS.items():
        csv_path = os.path.join(DIMENSIONS_PATH, f"{name_dim}.csv")
        if os.path.exists(csv_path):
            df_dim = pd.read_csv(csv_path, low_memory=False)
            
            # El ID siempre fue generado con este patrón en el script anterior
            id_column = f'ID_{name_dim.upper()}'
            
            # Filtramos para quedarnos estrictamente con el ID y las columns a cruzar (ignora las descripciones de texto aquí)
            necesary_column = [id_column] + columns
            dict_dims[name_dim] = df_dim[necesary_column]
        else:
            print(f"[Advertencia] No se encontró la dimensión {csv_path}")
            
    return dict_dims

In [4]:
def process_parquet_facts(dict_dims):
    """Itera sobre los CSV masivos, cruza con dimensiones, elimina columns originales y escribe Parquet incremental."""
    
    for origin_file, target_file in MAP_FILE.items():
        if not os.path.exists(origin_file):
            print(f"[Saltando] Archivo no encontrado: {origin_file}")
            continue
            
        print(f"\nIniciando transformación de: {origin_file}")
        
        # 1. Crear directorios de destino (Capa Gold) si no existen
        os.makedirs(os.path.dirname(target_file), exist_ok=True)
        
        # Objeto escritor de Parquet
        parquet_writer = None
        ok_rows = 0
        
        # 2. Lectura iterativa (Chunking)
        batch_iterator = pd.read_csv(origin_file, chunksize=250000, low_memory=False, encoding='latin1')
        
        for chunk in batch_iterator:
            ok_chunck = chunk.copy()
            
            # 3. Sustitución de Llaves (Key Surrogation)
            remove_columns = []
            
            for name_dim, columns_cruce in DIMENSIONS.items():
                if name_dim not in dict_dims: continue
                
                df_dim = dict_dims[name_dim]
                
                # Intersección: Solo cruzamos si el chunk actual tiene las columns
                cols_presents = [c for c in columns_cruce if c in ok_chunck.columns]
                
                if len(cols_presents) == len(columns_cruce):
                    # Realizamos el Left Join para traer el ID
                    ok_chunck = pd.merge(ok_chunck, df_dim, on=columns_cruce, how='left')
                    # Marcamos las columns originales para su destrucción
                    remove_columns.extend(columns_cruce)
            
            # 4. Destrucción de variables categóricas (Limpieza del Ciempiés)
            remove_columns = list(set(remove_columns)) # Eliminar duplicados en la lista por seguridad
            ok_chunck.drop(columns=remove_columns, inplace=True, errors='ignore')
            
            # 5. Optimización de Tipos de Datos (Manejo de nulos de Pandas en los IDs)
            # Al hacer Left Join, si alguna fila no cruzó perfectamente, Pandas convierte el ID a Float.
            # Convertimos todos los IDs a Int64 (Integer que soporta nulos) para que el Parquet sea perfecto.
            for_to_int = [col for col in ok_chunck.columns if col.startswith('ID_DIM_')]
            for col in for_to_int:
                ok_chunck[col] = ok_chunck[col].astype('Int64')
                
            # Las columns de fechas y ID_REGISTRO quedan intactas naturalmente al no haber sido eliminadas
            
            # 6. Escritura Incremental en Parquet
            table_pyarrow = pa.Table.from_pandas(ok_chunck)
            
            if parquet_writer is None:
                # Inicializa el archivo en el primer lote basándose en el esquema resultante
                parquet_writer = pq.ParquetWriter(target_file, table_pyarrow.schema, compression='snappy')
            
            parquet_writer.write_table(table_pyarrow)
            ok_rows += len(ok_chunck)
            print(f"  -> Lote procesado... ({ok_rows} filas insertadas)")
            
        # Cerrar el archivo de forma segura al terminar todos los lotes de ese año
        if parquet_writer:
            parquet_writer.close()
            
        print(f"[Éxito] Archivo final guardado en: {target_file}")

In [5]:
memory_dimensions = load_dimensions_in_memory()
process_parquet_facts(memory_dimensions)
print("\n[Operación Completada] Todos los datos han sido transformados a Esquema Estrella y exportados a la zona Gold en formato Parquet.")

Cargando dimensiones en memoria RAM...

Iniciando transformación de: ../DATA/Silver/COVID19MEXICO2020/COVID19MEXICO2020.csv
  -> Lote procesado... (99990 filas insertadas)
[Éxito] Archivo final guardado en: ../DATA/Gold/COVID19MEXICO2020/COVID19MEXICO2020.parquet

Iniciando transformación de: ../DATA/Silver/COVID19MEXICO2021/COVID19MEXICO2021.csv
  -> Lote procesado... (101342 filas insertadas)
[Éxito] Archivo final guardado en: ../DATA/Gold/COVID19MEXICO2021/COVID19MEXICO2021.parquet

Iniciando transformación de: ../DATA/Silver/COVID19MEXICO2022/COVID19MEXICO2022.csv
  -> Lote procesado... (99986 filas insertadas)
[Éxito] Archivo final guardado en: ../DATA/Gold/COVID19MEXICO2022/COVID19MEXICO2022.parquet

Iniciando transformación de: ../DATA/Silver/COVID19MEXICO2023/COVID19MEXICO2023.csv
  -> Lote procesado... (100610 filas insertadas)
[Éxito] Archivo final guardado en: ../DATA/Gold/COVID19MEXICO2023/COVID19MEXICO2023.parquet

Iniciando transformación de: ../DATA/Silver/COVID19MEXICO2